# E52 --- o teto que se demonstra, e o ponto em que o espectro cola

**A tentativa.** A secao anterior mediu os picos da norma da potencia onde o criterio do
autovalor fica cego (3,9 / 38,7 / 387,4 no nono dia). Falta o outro lado da moeda: se o
autovalor nao basta, que se de ao que ele nao ve um TETO demonstravel --- e o que a
demonstracao revela e que o teto e frouxo.

**O que se mede.**

1. a constante de Kreiss de cada coluna da familia, e o teto e (k+1) K contra o pico medido;
2. o desdobramento dos autovalores contra a abertura, em raiz, contra o controle simetrico, que
   desdobra linear;
3. a sensibilidade do autovalor, que diverge no ponto excepcional, e a fronteira (1-raio)^2/c;
4. a nuvem do raio espectral sob perturbacoes de norma fixa.

**Convencoes** (AGENTS.md paragrafos 7 e 9): um experimento por caderno, parametros no topo
marcados com "brinque com", algoritmo em frevolab, resultado em lab/resultados/E52_teto_e_ponto.json,
figuras em .pdf e .png.

In [1]:
# <- brinque com: ACOPLAMENTOS, ABERTURAS, PASSOS, RADIO, PONTOS, SORTES_NUVEM, ABERTURA_NUVEM, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

import frevolab
from frevolab import graficos, operador

ACOPLAMENTOS = (0.0, 1.0, 10.0, 100.0)   # a familia do capitulo: mesmo autovalor
ABERTURAS = tuple(10.0 ** k for k in (-3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0))
PASSOS = 200
RADIO = 0.9
PONTOS = 512
SORTES_NUVEM = 200
ABERTURA_NUVEM = 0.05
SEMENTE = 119                  # E40..E51 usam 107..118

print("frevolab %s | %d colunas | %d aberturas | semente %d"
      % (frevolab.VERSAO, len(ACOPLAMENTOS), len(ABERTURAS), SEMENTE))

frevolab 0.1.0 | 4 colunas | 7 aberturas | semente 119


## O teto demonstrável

A constante de Kreiss sai do resolvente, e o teto sai dela. O que se publica não é o teto: é a
folga --- quanto o limite fica acima do pico que a medição mostrou.

In [2]:
tetos = {}
for acoplamento in ACOPLAMENTOS:
    tetos[acoplamento] = operador.teto(acoplamento, passos=PASSOS, radio=RADIO, pontos=PONTOS)

print("%-8s %12s %8s %14s %12s %12s" % ("c", "Kreiss", "dia", "teto", "pico", "folga"))
for acoplamento, t in tetos.items():
    print("%-8.1f %12.2f %8d %14.2f %12.2f %12.2f"
          % (acoplamento, t["kreiss"], t["dia"], t["teto"], t["pico"], t["folga"]))

c              Kreiss      dia           teto         pico        folga
0.0              0.99        0           2.69         1.00         2.69
1.0              2.59        9          70.45         3.91        18.01
10.0            25.00        9         679.57        38.75        17.54
100.0          249.91        9        6793.14       387.42        17.53


In [3]:
# Figura 1: a norma da potencia dia a dia, com o teto tracejado.
fig, eixo = plt.subplots(figsize=(8.8, 4.2))
for acoplamento in ACOPLAMENTOS:
    serie = operador.normas(acoplamento, passos=PASSOS, radio=RADIO)
    eixo.plot(range(len(serie)), serie, "-", lw=1.6, label="c = %g" % acoplamento)
    eixo.axhline(tetos[acoplamento]["teto"], ls=":", lw=1.1, color="0.4")
eixo.set_yscale("log")
eixo.set_xlabel("dias")
eixo.set_ylabel("norma da potencia")
eixo.set_title("a influencia do estado inicial, e o teto que a limita", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E52_teto_e_ponto", 1)
plt.close(fig)
print("figura E52_teto_e_ponto_1 salva")

figura E52_teto_e_ponto_1 salva


## O ponto excepcional

Abrir o canto inferior separa os dois autovalores. A separação cresce na raiz da abertura --- e o
controle simétrico, que desdobra linear, cruza as retas da família.

In [4]:
ACOPLAMENTO_PONTO = 1.0
desdobramento = operador.desdobramento(ACOPLAMENTO_PONTO, ABERTURAS, radio=RADIO)
controle = [{"abertura": ab,
             "separacao": float(abs(np.diff(np.linalg.eigvals(operador.matriz_aberta(ab, ab, RADIO)))[0]))}
            for ab in ABERTURAS]
log_ab = np.log([l["abertura"] for l in desdobramento])
inclinacao = float(np.polyfit(log_ab, np.log([l["separacao"] for l in desdobramento]), 1)[0])
inclinacao_controle = float(np.polyfit(log_ab, np.log([l["separacao"] for l in controle]), 1)[0])
fracao_complexa = float(np.mean([1.0 if l["complexos"] else 0.0 for l in desdobramento]))
print("inclinacao da familia %.4f | do controle %.4f | fracao complexa %.3f"
      % (inclinacao, inclinacao_controle, fracao_complexa))

inclinacao da familia 0.5000 | do controle 1.0000 | fracao complexa 0.000


In [5]:
# Figura 2: a separacao dos autovalores contra a abertura, com o controle.
fig, eixo = plt.subplots(figsize=(8.4, 4.0))
eixo.plot([l["abertura"] for l in desdobramento], [l["separacao"] for l in desdobramento],
          "o-", color="#1f4e79", label="a familia (raiz da abertura)")
eixo.plot([l["abertura"] for l in controle], [l["separacao"] for l in controle],
          "s-", color="#c78f2c", label="o controle simetrico (linear)")
eixo.set_xscale("log")
eixo.set_yscale("log")
eixo.set_xlabel("a abertura")
eixo.set_ylabel("separacao dos autovalores")
eixo.set_title("o desdobramento em raiz, e o controle que desdobra linear", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E52_teto_e_ponto", 2)
plt.close(fig)
print("figura E52_teto_e_ponto_2 salva")

figura E52_teto_e_ponto_2 salva


In [6]:
# Figura 3: o raio espectral contra a abertura, com a fronteira e a nuvem.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
aberturas_finas = np.logspace(-4.5, 0.0, 90)
raios = [float(np.max(np.abs(np.linalg.eigvals(operador.matriz_aberta(ACOPLAMENTO_PONTO, ab, RADIO)))))
         for ab in aberturas_finas]
eixo.plot(aberturas_finas, raios, "-", color="#1f4e79", lw=1.6, label="o raio espectral")
nuvem = operador.nuvem(ACOPLAMENTO_PONTO, ABERTURA_NUVEM, SORTES_NUVEM, SEMENTE, radio=RADIO)
eixo.errorbar([ABERTURA_NUVEM], [float(np.median(nuvem))],
              yerr=[[float(np.median(nuvem) - np.quantile(nuvem, 0.05))],
                    [float(np.quantile(nuvem, 0.95) - np.median(nuvem))]],
              fmt="o", color="#b03a2e", capsize=4,
              label="a nuvem em abertura %g: %.1f%% acima de um" % (ABERTURA_NUVEM,
                                                                   100 * (nuvem > 1.0).mean()))
fronteira = operador.fronteira(ACOPLAMENTO_PONTO, radio=RADIO)
eixo.axvline(fronteira, color="#555555", ls="--", lw=1.3,
             label="a fronteira (1-raio)^2/c = %g" % fronteira)
eixo.axhline(1.0, color="0.3", ls=":", lw=1.1)
eixo.set_xscale("log")
eixo.set_xlabel("a abertura")
eixo.set_ylabel("raio espectral")
eixo.set_title("o raio sai do circulo na abertura prevista", fontsize=10)
eixo.legend(fontsize=7)
fig.tight_layout()
graficos.salvar(fig, "E52_teto_e_ponto", 3)
plt.close(fig)
print("figura E52_teto_e_ponto_3 salva")

figura E52_teto_e_ponto_3 salva


## Leitura visual das figuras

**Declarada contra os .png depois da execucao** (AGENTS.md paragrafo 9).

O que as legendas do capitulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: as quatro curvas com o pico, as linhas tracejadas do teto acima de todas elas, e a
   ordem das curvas acompanhando o acoplamento.
2. **Figura 2**: as duas retas em log-log, a da familia mais deitada (raiz) e a do controle com
   inclinacao maior (linear).
3. **Figura 3**: o raio subindo e cruzando a linha de um exatamente na vertical da fronteira, e a
   faixa da nuvem em torno do ponto marcado.

In [7]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
sens_pequena = operador.sensibilidade(ACOPLAMENTO_PONTO, ABERTURAS[0], radio=RADIO)
sens_grande = operador.sensibilidade(ACOPLAMENTO_PONTO, ABERTURAS[-2], radio=RADIO)
resultado = {
    "espectro_acoplamentos": len(ACOPLAMENTOS),
    "espectro_aberturas": len(ABERTURAS),
    "espectro_passos": PASSOS,
    "espectro_kreiss_familia": round(float(np.median([t["kreiss"] for t in tetos.values()])), 3),
    "espectro_kreiss_maior": round(max(t["kreiss"] for t in tetos.values()), 3),
    "espectro_pico_maior": round(max(t["pico"] for t in tetos.values()), 3),
    "espectro_teto_maior": round(max(t["teto"] for t in tetos.values()), 3),
    "espectro_folga_mediana": round(float(np.median([t["folga"] for t in tetos.values()])), 3),
    "espectro_fracao_do_pico": round(100 * float(np.mean(
        [t["pico"] / t["teto"] for t in tetos.values()])), 3),
    "espectro_dia_pico": int(max(t["dia"] for t in tetos.values())),
    "espectro_inclinacao_familia": round(inclinacao, 4),
    "espectro_inclinacao_controle": round(inclinacao_controle, 4),
    "espectro_fracao_complexa_pct": round(100 * fracao_complexa, 2),
    "espectro_fronteira_um": round(operador.fronteira(1.0, radio=RADIO), 5),
    "espectro_fronteira_cem": round(operador.fronteira(100.0, radio=RADIO), 7),
    "espectro_sensibilidade_pequena": round(sens_pequena, 3),
    "espectro_sensibilidade_grande": round(sens_grande, 3),
    "espectro_sensibilidade_razao": round(sens_pequena / sens_grande, 2),
    "espectro_nuvem_sortes": SORTES_NUVEM,
    "espectro_nuvem_mediana": round(float(np.median(nuvem)), 4),
    "espectro_nuvem_faixa_baixa": round(float(np.quantile(nuvem, 0.05)), 4),
    "espectro_nuvem_faixa_alta": round(float(np.quantile(nuvem, 0.95)), 4),
    "espectro_nuvem_acima_pct": round(100 * float((nuvem > 1.0).mean()), 2),
}
caminho = Path("lab/resultados/E52_teto_e_ponto.json")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1))

{
 "espectro_acoplamentos": 4,
 "espectro_aberturas": 7,
 "espectro_passos": 200,
 "espectro_kreiss_familia": 13.796,
 "espectro_kreiss_maior": 249.906,
 "espectro_pico_maior": 387.421,
 "espectro_teto_maior": 6793.144,
 "espectro_folga_mediana": 17.537,
 "espectro_fracao_do_pico": 13.528,
 "espectro_dia_pico": 9,
 "espectro_inclinacao_familia": 0.5,
 "espectro_inclinacao_controle": 1.0,
 "espectro_fracao_complexa_pct": 0.0,
 "espectro_fronteira_um": 0.01,
 "espectro_fronteira_cem": 0.0001,
 "espectro_sensibilidade_pequena": 15.811,
 "espectro_sensibilidade_grande": 0.889,
 "espectro_sensibilidade_razao": 17.78,
 "espectro_nuvem_sortes": 200,
 "espectro_nuvem_mediana": 1.1189,
 "espectro_nuvem_faixa_baixa": 0.9965,
 "espectro_nuvem_faixa_alta": 1.2062,
 "espectro_nuvem_acima_pct": 93.5
}
